In [1]:
import requests
from lxml import html
import json
import pandas as pd

from openpyxl import Workbook
from openpyxl.styles import Alignment


# Script to download data from the Customized Rainfall Information System (CRIS)
* The system by India's Meteorological Department, Ministry Of Earth Sciences
The main web site URL is: https://hydro.imd.gov.in/hydrometweb/(S(ly0bagfgqhfh2z55n1wy2u55))/DistrictRaifall.aspx

The site is setup so that one can select an Indian state and subsequently an associated district and a page will display an HTML table of rainfall
by year and month.

This scripts loops through each state and associated districts to scrape the HTML rainfall table from the site and save the data into an Excel file

* Note: the web site uses page redirects often so it is important to follow the redirects when submitting POST requests

## Setup global variables/objects

In [2]:
MAX_REDIRECTS = 5 #max number of times to follow a page redirect

# Define helper functions

In [3]:
# submit state or state/district options to web site and get response
# If the web site redirects to another page, follow the redirection
# and try to re-post the data to the new page
# will only retry up to MAX_REDIRECTS number of times
def submit_post_request(url, data):
    response = requests.post(url, data = data, allow_redirects=False)
    i = MAX_REDIRECTS
    while i > 0 and response.status_code in (301, 302):
        print(f"following redirect. {MAX_REDIRECTS} redirects remain")
        response = requests.post(response.headers['Location'], data = data, allow_redirects=False)
        i -= 1    
        
    return response

# parses the data returned by the main page
# This script evaluates the SELECT HTML element with id "listItems"
# which contains the values of all the states
# It returns the POST form data that is used to send to the site
# to get the list of associated districts
# This function is intended to be used as a Python generator
def state_get(page):
    tree = html.fromstring(page.content)
    view_state = tree.xpath("//input[@id='__VIEWSTATE']")[0].value
    
    states = tree.xpath("//select[@id='listItems']")[0]
    for state in states.value_options[1:]:
        formdata = {
            'listItems' : state,
            '__EVENTTARGET': 'listItems',
            '__VIEWSTATE' : view_state,
             '__EVENTARGUMENT' : '',
             '__LASTFOCUS: ' : '',            
        }
        yield formdata

        
        
# parses the data returned by the page for a specific state
# This script evaluates the SELECT HTML element with id "DistrictDropDownList"
# which contains the values of all the disitrcts for a given state
# It returns the POST form data that is used to send to the site
# to get the rainfall data for a given district
# This function is intended to be used as a Python generator        
def district_get(state, page):
    tree = html.fromstring(page.content)
    view_state = tree.xpath("//input[@id='__VIEWSTATE']")[0].value
    
    districts = tree.xpath("//select[@id='DistrictDropDownList']")[0]
    for district in districts.value_options[1:]:
        formdata = {
            'listItems' : state,
            'DistrictDropDownList' : district,
            'GoBtn' : "GO",
            '__EVENTTARGET': '',
            '__VIEWSTATE' : view_state,
             '__EVENTARGUMENT' : '',
             '__LASTFOCUS: ' : '',            
        }
        yield formdata   
        
        
# parses the HTML page thhat contains rainfall data from a specific district
# this function searches for an HTML tabled with the id "GridId" which
# contains the rainfall data
# it skips the header rows and saves each table cell data into a python list
# If it cannot find the table (because there is no data for the district), the it retuns an empty list
def parse_table(state, district, page):
    table_data = []
    
    tree = html.fromstring(page.content)
    tables = tree.xpath("//table[@id='GridId']")
    if len (tables) == 0:
        return []
    
    table = tables[0]

    for row in table.xpath(".//tr")[3:]:  #skip first 3 rows because they're headers
        row_data = [state, district]
        for col in row.xpath(".//td")[1:]: #skip first column because it's blank
            cell_value = col.xpath(".//text()")[0].replace("\r\n", "").strip()
            row_data.append(cell_value)
        table_data.append(row_data)
    
        
    return table_data
    
def is_even(number):
    if (number % 2) == 0:
        return True
    else:
        return False
    
# this routine takes a list of the combined rainfall data for all districts and saves to an Excel file
def save_to_excel(filename, table_data):

    wb = Workbook()
    ws = wb.active

    
    # row column layout

        #header first row
        # state : row 1, col 1
        # district : row 1, col 2
        # year : row 1, col 3
        # JAN : row 1, cols 4-5
        # FEV : row 1, cols 6-7
        # ...
        # DEC : row 1, cols 26-27

        #header second row
        # R/F : row 2, starting on col 4 (every even column) to col 26
        # % DEP : row 2, starting on col 5 (every odd column) to col 27
        
        
        # the first datarow starts on row 3



    ws.title = "Rain Data"

    #Add header row - First had 3 columns for state, district, and year
    row_month = 1
    ws.cell(row=row_month, column=1).value = "State"
    ws.cell(row=row_month, column=2).value = "District"
    ws.cell(row=row_month, column=3).value = "Year"
    ws.cell(row=row_month, column=1).alignment = Alignment(horizontal='center')
    ws.cell(row=row_month, column=2).alignment = Alignment(horizontal='center')
    ws.cell(row=row_month, column=3).alignment = Alignment(horizontal='center')
    month_start_col = 4

    #now add the month, and merge cells
    months = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL",  "AUG", "SEPT", "OCT", "NOV", "DEC"]
    for i in range(len(months)):
        col_month = (i * 2) + month_start_col
        col_rf = col_month
        col_dp = col_month + 1

        ws.cell(row=row_month, column=col_month).value = months[i]
        ws.cell(row=row_month, column=col_month).alignment = Alignment(horizontal='center')
        ws.merge_cells(start_row=row_month, start_column=col_month, end_row=row_month, end_column=col_month+1)

    #add second header row
    row_rf_dep = 2
    for i in range(len(months)):
        col_rf = (i * 2) + month_start_col
        col_dep = col_rf + 1
        ws.cell(row=row_rf_dep, column=col_rf).value = "R/F"
        ws.cell(row=row_rf_dep, column=col_rf).alignment = Alignment(horizontal='center')
        ws.cell(row=row_rf_dep, column=col_dep).value = "%DEP"
        ws.cell(row=row_rf_dep, column=col_dep).alignment = Alignment(horizontal='center')


    row_data = 3   
    for row in table_data:
        for i in range(len(row)):
            cell_text = row[i]

            if i < 3: #state, district, year
                cell_value = cell_text
            
            elif cell_text == "": #somtimes data for that month is blank
                cell_value = cell_text                
            
            #this  s %DEP
            #sometimes text is in 23.5% and sometimes in 23.5
            elif is_even(i): 
                if cell_text[-1] == "%": #convert text with % to decimal
                    cell_text = cell_text.replace('%', 'e-2')
                else: 
                    cell_text = cell_text + 'e-2' # convert to decimal without %
                cell_value = float(cell_text)
            
            else:
                cell_value = float(cell_text) #convert rainfall data to decimal
            ws.cell(row=row_data, column=i+1).value = cell_value

        row_data += 1



    wb.save(filename)

In [5]:
table_data = [] # this contains the data for all the districts


output_excel = "indian_rain_data.xlsx"   #name of file. will be saved in the same location as this notebook

#first page to go to to get the list of states
page_url = "https://hydro.imd.gov.in/hydrometweb/(S(ex5xjh55hjeddy55olttmp55))/DistrictRaifall.aspx"

print("Getting list of states...", end ='')
main_page = requests.get(page_url)
states_url = main_page.url #updated URL based on page redirection
print("done")

for state_data in state_get(main_page):
    # print(json.dumps(state_data, indent=1))
    state = state_data["listItems"]
    print("Getting districts for ", state)
    state_page = submit_post_request(states_url, data = state_data)
    districts_url = state_page.url #updated URL based on page redirection    
    
    # print(state_page.content)
    for district_data in district_get(state, state_page):
        district = district_data["DistrictDropDownList"]
        print("--downloading data for ", district)
        # print(f"Getting data {state} -> {district}" )
        district_page = submit_post_request(districts_url, data = district_data)
        # page_url = district_page.url #updated URL based on page redirection  
        
        district_table_data = parse_table(state, district, district_page)
        if len(district_table_data) == 0:
            print(f"no data for {state} > {district}")
        table_data.extend(district_table_data)


print("saving data to Excel...", end = '')
save_to_excel(output_excel, table_data)
print(f"file saved to {output_excel}")
        
        
    
        


Getting list of states...done
Getting districts for  A & N ISLAND (UT)
--downloading data for  NICOBAR
--downloading data for  NORTH & MIDDLE ANDAMAN
--downloading data for  SOUTH ANDAMAN
Getting districts for  ANDHRA PRADESH
--downloading data for  ANANTAPURAMU
--downloading data for  CHITTOOR
--downloading data for  EAST GODAVARI
--downloading data for  GUNTUR
--downloading data for  KRISHNA
--downloading data for  KURNOOL
--downloading data for  PRAKASAM
--downloading data for  SPSR NELLORE
--downloading data for  SRIKAKULAM
--downloading data for  VISHAKHAPATNAM
--downloading data for  VIZIANAGARAM
--downloading data for  WEST GODAVARI
--downloading data for  YSR DISTRICT
Getting districts for  ARUNACHAL PRADESH
--downloading data for  ANJAW
--downloading data for  CHANGLANG
--downloading data for  DIBANG VALLEY
--downloading data for  EAST KAMENG
--downloading data for  EAST SIANG
--downloading data for  KURUNG KUMEY
--downloading data for  LOHIT
--downloading data for  LOWER DIBA